### In this notebook we load in the gdp dataset and generate csv files for undrogued beaching, undrogued not beaching, drogued beaching, drogued not beaching, the full set beaching, and the full set not beaching. We separate drogued from undrogued, beaching from never beaching, and calculate the time to beach variable for those drifters that beach.

In [1]:
# set working directory
import os
os.chdir('/dat1/openonic/Drifters') # directory

In [2]:
# dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import time

In [3]:
# load file
gdp = xr.open_dataset('gdp.nc', decode_times=False)

In [4]:
# lets take a look at the data
gdp

<xarray.Dataset> Size: 13GB
Dimensions:                (traj: 17324, obs: 165754333)
Coordinates:
    ID                     (traj) int64 139kB ...
    longitude              (obs) float32 663MB ...
    latitude               (obs) float32 663MB ...
    time                   (obs) int64 1GB ...
    ids                    (obs) int64 1GB ...
Dimensions without coordinates: traj, obs
Data variables: (12/53)
    rowsize                (traj) int64 139kB ...
    location_type          (traj) bool 17kB ...
    WMO                    (traj) int32 69kB ...
    expno                  (traj) int32 69kB ...
    deploy_date            (traj) float64 139kB ...
    deploy_lon             (traj) float32 69kB ...
    ...                     ...
    err_sst                (obs) float32 663MB ...
    err_sst1               (obs) float32 663MB ...
    err_sst2               (obs) float32 663MB ...
    flg_sst                (obs) int8 166MB ...
    flg_sst1               (obs) int8 166MB ...
    flg_sst2               (obs) int8 166MB ...
Attributes: (12/15)
    title:             Global Drifter Program hourly drifting buoy collection
    history:           Version 2.00.  Metadata from dirall.dat and deplog.dat
    Conventions:       CF-1.6
    date_created:      2022-01-14T11:16:01.998226
    publisher_name:    GDP Drifter DAC
    publisher_email:   aoml.dftr@noaa.gov
    ...                ...
    metadata_link:     https://www.aoml.noaa.gov/phod/dac/dirall.html
    contributor_name:  NOAA Global Drifter Program
    contributor_role:  Data Acquisition Center
    institution:       NOAA Atlantic Oceanographic and Meteorological Laboratory
    acknowledgement:   Elipot et al. (2022) to be submitted. Elipot et al. (2...
    summary:           Global Drifter Program hourly data

In [5]:
start_time = time.time()

ids = gdp.ids.values.astype(np.float32)
times = gdp.time.values.astype(np.float32)
lats = gdp.latitude.values.astype(np.float32)
lons = gdp.longitude.values.astype(np.float32)
ves = gdp.ve.values.astype(np.float32)
vns = gdp.vn.values.astype(np.float32)

full_data = pd.DataFrame({
    'id': ids,
    'time': times,
    'lat': lats.astype(np.float32),
    'lon': lons.astype(np.float32),
    've': ves.astype(np.float32),
    'vn': vns.astype(np.float32)
})

elapsed_time = time.time() - start_time
print(f"Elapsed time: {elapsed_time} seconds")

Elapsed time: 13.454550504684448 seconds


In [6]:
# SEPARATE BEACHED / UNBEACHED TRAJECTORY IDS

In [7]:
beach_ids = []
unbeach_ids = []

for i in range(len(gdp.type_death)):
    if gdp.type_death[i].item() == 1:
        beach_ids.append(gdp.ID[i].item())
    else:
        unbeach_ids.append(gdp.ID[i].item())

In [8]:
# full set
beach_full = full_data[full_data['id'].isin(beach_ids)]
unbeach_full = full_data[full_data['id'].isin(unbeach_ids)]


In [9]:
# DROGUE LOST DATE INFO

In [10]:
min_time = gdp.drogue_lost_date.values
min_ID = gdp.drogue_lost_date.coords['ID'].values
min_time_df = pd.DataFrame({'min_time': min_time, 'id': min_ID})

In [11]:
# end date / location info (for later recovery)
traj_ID_vals = gdp.ID.values
end_time_vals = gdp.end_date.values
end_lat_vals = gdp.end_lat.values
end_lon_vals = gdp.end_lon.values
end_time_df = pd.DataFrame({
    'id': traj_ID_vals,
    'end_time': end_time_vals,
    'end_lat': end_lat_vals,
    'end_lon': end_lon_vals
})

In [12]:
# CLASSIFY DRIFTERS: DROGUED BEACHING vs UNDROGUED BEACHING

In [13]:
# ============================================================
# CLASSIFY ALL 17,324 DRIFTERS INTO 4 MUTUALLY EXCLUSIVE GROUPS
# 
# "Drogued" = never lost drogue, OR lost it within 1 day of 
#             beaching, OR drogue_lost_date >= end_date
# "Undrogued" = lost drogue >1 day before end of record
# "Beaching" = type_death == 1
# "Non-beaching" = type_death != 1

In [14]:
all_traj_ids = set(gdp.ID.values)
beach_ids_set = set(beach_ids)
unbeach_ids_set = set(unbeach_ids)

In [15]:
# --- Identify drifters that never lost their drogue (NaN drogue_lost_date) ---
never_lost_drogue_ids = set(gdp.ID.values[np.isnan(gdp.drogue_lost_date.values)])
print(f"Never lost drogue (NaN drogue_lost_date): {len(never_lost_drogue_ids)}")

# --- Identify drifters where drogue_lost_date >= end_date ---
erroneous_mask = (gdp.drogue_lost_date.values >= gdp.end_date.values)
erroneous_ids = set(gdp.ID.values[erroneous_mask & ~np.isnan(gdp.drogue_lost_date.values)])
print(f"drogue_lost_date >= end_date: {len(erroneous_ids)}")

Never lost drogue (NaN drogue_lost_date): 142
drogue_lost_date >= end_date: 5510


In [16]:
# --- Identify beached drifters within 1 day of drogue loss ---
beach_end_drogue = pd.merge(
    end_time_df[end_time_df['id'].isin(beach_ids)][['id', 'end_time']],
    min_time_df[min_time_df['id'].isin(beach_ids)],
    on='id', how='inner'
)
beach_end_drogue['time_diff'] = beach_end_drogue['end_time'] - beach_end_drogue['min_time']
within_1day_ids = set(beach_end_drogue[
    (beach_end_drogue['time_diff'] >= 0) & 
    (beach_end_drogue['time_diff'] <= 86400)  # 1 day in seconds
]['id'].values)
print(f"Beached within 1 day of drogue loss: {len(within_1day_ids)}")

Beached within 1 day of drogue loss: 1102


In [17]:
# --- "Effectively drogued" = never lost drogue OR erroneous OR within 1 day ---
effectively_drogued_beach = (never_lost_drogue_ids | erroneous_ids | within_1day_ids) & beach_ids_set
effectively_drogued_unbeach = never_lost_drogue_ids & unbeach_ids_set

effectively_drogued_ids = effectively_drogued_beach | effectively_drogued_unbeach
effectively_undrogued_ids = all_traj_ids - effectively_drogued_ids

# --- 4 mutually exclusive groups ---
drogued_beach_ids = effectively_drogued_ids & beach_ids_set
drogued_unbeach_ids = effectively_drogued_ids & unbeach_ids_set
undrogued_beach_ids = effectively_undrogued_ids & beach_ids_set
undrogued_unbeach_ids = effectively_undrogued_ids & unbeach_ids_set

print(f"\n=== CLASSIFICATION ===")
print(f"Drogued beaching:       {len(drogued_beach_ids)}")
print(f"Drogued non-beaching:   {len(drogued_unbeach_ids)}")
print(f"Undrogued beaching:     {len(undrogued_beach_ids)}")
print(f"Undrogued non-beaching: {len(undrogued_unbeach_ids)}")
print(f"Total:                  {len(drogued_beach_ids) + len(drogued_unbeach_ids) + len(undrogued_beach_ids) + len(undrogued_unbeach_ids)}")
print(f"Overlap check:          {len(effectively_drogued_ids & effectively_undrogued_ids)}")


=== CLASSIFICATION ===
Drogued beaching:       1102
Drogued non-beaching:   142
Undrogued beaching:     3291
Undrogued non-beaching: 12789
Total:                  17324
Overlap check:          0


In [18]:
# TIME TO BEACH FUNCTION

In [19]:
def find_time_to_beach(beach, beach_last, time_between_register_beaching_and_actually_beaching):
    beach_array = beach.to_numpy()
    beach_last_array = beach_last.to_numpy()
    last_time_dict = dict(zip(beach_last_array[:, 0], beach_last_array[:, 1]))
    beach_time = []

    for row in beach_array:
        current_ID = row[0]
        last_time = last_time_dict.get(current_ID, None)

        if last_time is not None:
            current_time = row[1]
            if current_time != last_time:
                time_difference = last_time - current_time
                beach_time.append(time_difference)
            if current_time == last_time:
                beach_time.append(time_between_register_beaching_and_actually_beaching)

    return beach_time

time_between_register_beaching_and_actually_beaching = 0

In [20]:
# UNDROGUED BEACHED
# Post-drogue-loss observations for drifters that beached >1 day after losing drogue

In [21]:
beach_min_time_undrogued = min_time_df[min_time_df['id'].isin(undrogued_beach_ids)]
merged_beach = pd.merge(
    beach_full[beach_full['id'].isin(undrogued_beach_ids)], 
    beach_min_time_undrogued, 
    left_on='id', right_on='id', how='inner'
)
undrogued_beach_ = merged_beach[merged_beach['time'] >= merged_beach['min_time']]
undrogued_beach = undrogued_beach_.drop(columns=['min_time'])

# recover missing drifters (0 obs after drogue loss)
missing_ub_ids = undrogued_beach_ids - set(undrogued_beach['id'].unique())
if len(missing_ub_ids) > 0:
    print(f"Recovering {len(missing_ub_ids)} undrogued beach drifters with 0 post-drogue obs")
    missing_data = end_time_df[end_time_df['id'].isin(missing_ub_ids)].copy()
    missing_drogue = min_time_df[min_time_df['id'].isin(missing_ub_ids)]
    missing_merged = pd.merge(missing_data, missing_drogue, on='id', how='inner')
    missing_rows = pd.DataFrame({
        'id': missing_merged['id'],
        'time': missing_merged['end_time'],
        'lat': missing_merged['end_lat'].astype(np.float32),
        'lon': missing_merged['end_lon'].astype(np.float32),
        've': np.float32(0.0),
        'vn': np.float32(0.0)
    })
    undrogued_beach = pd.concat([undrogued_beach, missing_rows], ignore_index=True)

# calculate time_to_beach (time from each obs to last obs for that trajectory)
last_points = undrogued_beach.drop_duplicates(subset='id', keep='last')
beach_time = find_time_to_beach(undrogued_beach, last_points, time_between_register_beaching_and_actually_beaching)
undrogued_beach['time_to_beach'] = beach_time

print(f"Undrogued beach rows: {len(undrogued_beach)}, unique IDs: {undrogued_beach['id'].nunique()}")

Recovering 953 undrogued beach drifters with 0 post-drogue obs
Undrogued beach rows: 17629438, unique IDs: 3291


In [22]:
# UNDROGUED UNBEACHED

In [23]:
unbeach_min_time_undrogued = min_time_df[min_time_df['id'].isin(undrogued_unbeach_ids)]
merged_unbeach = pd.merge(
    unbeach_full[unbeach_full['id'].isin(undrogued_unbeach_ids)],
    unbeach_min_time_undrogued,
    left_on='id', right_on='id', how='inner'
)
undrogued_unbeach_ = merged_unbeach[merged_unbeach['time'] >= merged_unbeach['min_time']]
undrogued_unbeach = undrogued_unbeach_.drop(columns=['min_time'])

# recover missing drifters (0 obs after drogue loss)
missing_uub_ids = undrogued_unbeach_ids - set(undrogued_unbeach['id'].unique())
if len(missing_uub_ids) > 0:
    print(f"Recovering {len(missing_uub_ids)} undrogued unbeach drifters with 0 post-drogue obs")
    missing_data = end_time_df[end_time_df['id'].isin(missing_uub_ids)].copy()
    missing_rows = pd.DataFrame({
        'id': missing_data['id'],
        'time': missing_data['end_time'],
        'lat': missing_data['end_lat'].astype(np.float32),
        'lon': missing_data['end_lon'].astype(np.float32),
        've': np.float32(0.0),
        'vn': np.float32(0.0)
    })
    undrogued_unbeach = pd.concat([undrogued_unbeach, missing_rows], ignore_index=True)

print(f"Undrogued unbeach rows: {len(undrogued_unbeach)}, unique IDs: {undrogued_unbeach['id'].nunique()}")

MemoryError: Unable to allocate 763. MiB for an array with shape (99977711,) and data type int64

In [ ]:
# DROGUED BEACHED
# Full trajectory (all observations) for drifters that beached WITH drogue

In [ ]:
drogued_beach = beach_full[beach_full['id'].isin(drogued_beach_ids)].copy()

last_points = drogued_beach.drop_duplicates(subset='id', keep='last')
beach_time = find_time_to_beach(drogued_beach, last_points, time_between_register_beaching_and_actually_beaching)
drogued_beach['time_to_beach'] = beach_time

print(f"Drogued beach rows: {len(drogued_beach)}, unique IDs: {drogued_beach['id'].nunique()}")

In [ ]:
# RECLASSIFY: move any undrogued_beach with max time_to_beach 
# <= 1 day into drogued_beach (edge cases where drogue_lost_date
# is slightly >1 day before end_date but first undrogued obs
# is within 1 day of end)
max_ttb_check = undrogued_beach.groupby('id')['time_to_beach'].max().reset_index()
reclassify_ids = set(max_ttb_check[max_ttb_check['time_to_beach'] <= 86400]['id'].values)

if len(reclassify_ids) > 0:
    print(f"Reclassifying {len(reclassify_ids)} drifters from undrogued_beach to drogued_beach")
    
    # move their FULL trajectories to drogued_beach
    reclassified_rows = beach_full[beach_full['id'].isin(reclassify_ids)].copy()
    last_pts = reclassified_rows.drop_duplicates(subset='id', keep='last')
    rc_beach_time = find_time_to_beach(reclassified_rows, last_pts, time_between_register_beaching_and_actually_beaching)
    reclassified_rows['time_to_beach'] = rc_beach_time
    drogued_beach = pd.concat([drogued_beach, reclassified_rows], ignore_index=True)
    
    # remove from undrogued_beach
    undrogued_beach = undrogued_beach[~undrogued_beach['id'].isin(reclassify_ids)].reset_index(drop=True)
    
    # update id sets
    drogued_beach_ids = drogued_beach_ids | reclassify_ids
    undrogued_beach_ids = undrogued_beach_ids - reclassify_ids

In [ ]:
# DROGUED UNBEACHED
# Pre-drogue-loss observations for drifters that never beached

In [ ]:
drogued_unbeach = unbeach_full[unbeach_full['id'].isin(drogued_unbeach_ids)].copy()
print(f"Drogued unbeach rows: {len(drogued_unbeach)}, unique IDs: {drogued_unbeach['id'].nunique()}")


In [ ]:
# FULL SET TIME TO BEACH

In [ ]:
last_points_full = beach_full.drop_duplicates(subset='id', keep='last')
beach_time_full = find_time_to_beach(beach_full, last_points_full, time_between_register_beaching_and_actually_beaching)
beach_full['time_to_beach'] = beach_time_full

In [ ]:
# VERIFICATION

In [ ]:
print(f"\n=== VERIFICATION ===")
n_undrogued_beach = undrogued_beach['id'].nunique()
n_undrogued_unbeach = undrogued_unbeach['id'].nunique()
n_drogued_beach = drogued_beach['id'].nunique()
n_drogued_unbeach = drogued_unbeach['id'].nunique()
total = n_undrogued_beach + n_undrogued_unbeach + n_drogued_beach + n_drogued_unbeach

print(f"Undrogued beaching drifters:     {n_undrogued_beach} (should be {len(undrogued_beach_ids)})")
print(f"Undrogued non-beaching drifters: {n_undrogued_unbeach} (should be {len(undrogued_unbeach_ids)})")
print(f"Drogued beaching drifters:       {n_drogued_beach} (should be {len(drogued_beach_ids)})")
print(f"Drogued non-beaching drifters:   {n_drogued_unbeach} (should be {len(drogued_unbeach_ids)})")
print(f"Total: {total} (should be 17324)")

all_undrogued = set(undrogued_beach['id'].unique()) | set(undrogued_unbeach['id'].unique())
all_drogued = set(drogued_beach['id'].unique()) | set(drogued_unbeach['id'].unique())
print(f"Overlap drogued/undrogued:       {len(all_undrogued & all_drogued)} (should be 0)")


In [ ]:
# How many of erroneous_ids are non-beaching?
erroneous_nonbeach = erroneous_ids & unbeach_ids_set
erroneous_beach = erroneous_ids & beach_ids_set
print(f"erroneous_ids that are non-beaching: {len(erroneous_nonbeach)}")
print(f"erroneous_ids that are beaching: {len(erroneous_beach)}")

In [ ]:
# SAVE TO CSV

In [34]:
print('saving undrogued_beach')
undrogued_beach.to_csv('undrogued_beach.csv', index=False)
print('saving undrogued_unbeach')
undrogued_unbeach.to_csv('undrogued_unbeach.csv', index=False)
print('saving drogued_beach')
drogued_beach.to_csv('drogued_beach.csv', index=False)
print('saving drogued_unbeach')
drogued_unbeach.to_csv('drogued_unbeach.csv', index=False)
print('saving full_beach')
beach_full.to_csv('full_beach.csv', index=False)
print('saving full_unbeach')
unbeach_full.to_csv('full_unbeach.csv', index=False)
print('done!')

saving undrogued_beach
saving undrogued_unbeach
saving drogued_beach
saving drogued_unbeach
saving full_beach
saving full_unbeach
done!
